In [2]:
import torch
import torch.nn as nn

In [3]:
import zipfile

zip_path = r"C:\Users\yapri\Downloads\archive (2).zip"
extract_path = r"C:\Users\yapri\Downloads\catsvsdogs_data"

with zipfile.ZipFile(zip_path, "r") as zip_r:
    zip_r.extractall(extract_path)

In [4]:
from torchvision import datasets,transforms
device=torch.device("cuda")
train_path=r"C:\Users\yapri\Downloads\catsvsdogs_data\catsvsdogs\train"
test_path=r"C:\Users\yapri\Downloads\catsvsdogs_data\catsvsdogs\test"
train_transform=transforms.Compose([transforms.Resize((256,256)),
                                    transforms.RandomHorizontalFlip(p=0.5),
                                    transforms.RandomRotation(degrees=15),
                                    transforms.ColorJitter(brightness=0.2,contrast=0.2,saturation=0.2),
                                    transforms.ToTensor()])
test_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor()])
train_dataset=datasets.ImageFolder(root=train_path,transform=train_transform)
test_dataset=datasets.ImageFolder(root=test_path,transform=test_transform)

In [5]:
print("Train images:", len(train_dataset))
print("Test images:", len(test_dataset))
print("Classes:", train_dataset.classes)
print("Class mapping:", train_dataset.class_to_idx)

Train images: 20000
Test images: 5000
Classes: ['cats', 'dogs']
Class mapping: {'cats': 0, 'dogs': 1}


In [6]:
from torch.utils.data import  Dataset, DataLoader
train_loader=DataLoader(train_dataset,
                        batch_size=32,
                        shuffle=True,
                        pin_memory=True,
                        num_workers=2)
test_loader = DataLoader(test_dataset,
                         batch_size=32,
                         shuffle=False,
                         num_workers=2,
                         pin_memory=True)

In [7]:
# to check data is properly fetched
images, labels = next(iter(train_loader))
print("Images shape:", images.shape)
print("Labels shape:", labels.shape)
print("Labels:", labels[:10])
images = images.to(device, non_blocking=True)
labels = labels.to(device, non_blocking=True)
print("Image device:", images.device)
print("Label device:", labels.device)

Images shape: torch.Size([32, 3, 256, 256])
Labels shape: torch.Size([32])
Labels: tensor([1, 0, 1, 0, 0, 1, 0, 0, 0, 1])
Image device: cuda:0
Label device: cuda:0


In [8]:
class catdogcnn(nn.Module):
    def __init__(self,channel):
        super().__init__()
        self.feature=nn.Sequential(nn.Conv2d(channel,32,kernel_size=3,padding=1),
                                   nn.ReLU(),
                                   nn.MaxPool2d(kernel_size=2,stride=2),
                                   nn.Conv2d(32,64,kernel_size=3,padding=1),
                                   nn.ReLU(),
                                   nn.MaxPool2d(kernel_size=2,stride=2),
                                   nn.Conv2d(64,128,kernel_size=3,padding=1),
                                   nn.ReLU(),
                                   nn.MaxPool2d(kernel_size=2,stride=2))
        self.classifier=nn.Sequential(nn.Flatten(),nn.Linear(32*32*128,128),
                                     nn.ReLU(),
                                     nn.Dropout(0.3),
                                     nn.Linear(128,1))
    def forward(self,x):
            x=self.feature(x)
            x=self.classifier(x)
            return x       

In [9]:
model=catdogcnn(3)
model=model.to(device)

In [10]:
epochs=20
loss_fun=nn.BCEWithLogitsLoss()
optimizer=torch.optim.Adam(model.parameters(),lr=0.001)

In [11]:
for epoch in range(epochs):
    total_loss = 0
    for batch_feature, batch_label in train_loader:
        batch_feature = batch_feature.to(device)
        batch_label = batch_label.to(device)
        y_pred = model(batch_feature)
        loss = loss_fun(
            y_pred,
            batch_label.view(-1, 1).float() )
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    average_loss = total_loss / len(train_loader)
    print( f"Epoch {epoch + 1}/{epochs} "f"- Loss: {average_loss:.4f}")

Epoch 1/20 - Loss: 0.6794
Epoch 2/20 - Loss: 0.6008
Epoch 3/20 - Loss: 0.5466
Epoch 4/20 - Loss: 0.4969
Epoch 5/20 - Loss: 0.4688
Epoch 6/20 - Loss: 0.4481
Epoch 7/20 - Loss: 0.4319
Epoch 8/20 - Loss: 0.4105
Epoch 9/20 - Loss: 0.3941
Epoch 10/20 - Loss: 0.3835
Epoch 11/20 - Loss: 0.3681
Epoch 12/20 - Loss: 0.3541
Epoch 13/20 - Loss: 0.3419
Epoch 14/20 - Loss: 0.3406
Epoch 15/20 - Loss: 0.3306
Epoch 16/20 - Loss: 0.3214
Epoch 17/20 - Loss: 0.3157
Epoch 18/20 - Loss: 0.3067
Epoch 19/20 - Loss: 0.2977
Epoch 20/20 - Loss: 0.2903


In [ ]:

model.eval()
for epoch in range(10):
    val_loss=0
    with torch.no_grad():
        for batch_feature,batch_label in test_loader:
            batch_feature,batch_label=batch_feature.to(device),batch_label.to(device)
            output=model.forward(batch_feature)
            loss=loss_fun(output,batch_label.view(-1,1).float())
            val_loss+=loss.item()
    avg_val_loss=val_loss/len(test_loader)
    print(f"Epoch {epoch}: Validation Loss = {avg_val_loss:.4f}")

Epoch 0: Validation Loss = 0.3143
Epoch 1: Validation Loss = 0.3143
Epoch 2: Validation Loss = 0.3143
Epoch 3: Validation Loss = 0.3143
Epoch 4: Validation Loss = 0.3143
Epoch 5: Validation Loss = 0.3143
Epoch 6: Validation Loss = 0.3143
Epoch 7: Validation Loss = 0.3143
Epoch 8: Validation Loss = 0.3143


In [ ]:
model.eval()
correct = 0
total = 0
with torch.no_grad():
    for batch_feature, batch_label in test_loader:
        batch_feature = batch_feature.to(device)
        batch_label = batch_label.to(device)
        output = model(batch_feature)
        prediction = (torch.sigmoid(output) >= 0.5).long()
        correct += (prediction.view(-1) == batch_label).sum().item()
        total += batch_label.size(0)
accuracy = correct / total
print(f"Test Accuracy: {accuracy * 100:.2f}%")

In [ ]:
 from PIL import Image 
import torch 
image_path=r"C:\Users\yapri\Downloads\cat_1.webp"
image=Image.open(image_path).convert("RGB")
image=test_transform(image)
image=image.unsqueeze(0)
image=image.to(device)
model.eval()
with torch.no_grad():
    output=model.forward(image)
    probability=torch.sigmoid(output).item()
    if probability >=0.7:
        prediction="dog"
    elif probability<=0.2:
        prediction="cat"
    else:
        prediction="not confident"
print("Prediction:", prediction)
print(f"Dog Probability: {probability * 100:.2f}%")
print(f"Cat Probability: {(1 - probability) * 100:.2f}%")    

In [ ]:
torch.save(model.state_dict(), "cat_dog_cnn.pth")